In [2]:
suppressMessages({library("rwwa"); library(quantreg)})

gmst <- read.table("data/gmst.dat", col.names = c("year", "gmst"))
gmst$gmst <- gmst$gmst - gmst$gmst[gmst$year == 2026]

## Check data availability

In [3]:
t((sapply(list.files("stations", full.names = T), function(fnm) {
    ts <- read.csv(fnm)[,1:2]
    c("prop_missing" = sum(is.na(ts)) / nrow(ts) * 100,
      "nobs" = nrow(ts) / 365)
    })))

,prop_missing,nobs
stations/A_CORUNA_ALVEDRO_SP_LAT43.3067_LON-8.3719.csv,8.9395090,54.12329
stations/A_CORUNA_SP_LAT43.3669_LON-8.4192.csv,4.9088198,47.77534
stations/BADAJOZ_TALAVERA_LA_REAL_SP_LAT38.8831_LON-6.8292.csv,0.4588748,71.04932
stations/CADIZ_SP_LAT36.5008_LON-6.2567.csv,2.7528861,33.93699
stations/CIUDAD_REAL_SP_LAT38.9892_LON-3.9194.csv,5.3748512,27.62740
stations/CORDOBA_AEROPUERTO_SP_LAT37.8442_LON-4.8458.csv,2.0672047,33.26575
stations/GRANADA_AEROPUERTO_SP_LAT37.1894_LON-3.7892.csv,2.4100092,26.71507
stations/GRAZALEMA_SP_LAT36.7583892_LON-5.366074.csv,0.0000000,24.70959
stations/HUELVA_RONDA_DEL_ESTE_SP_LAT37.28_LON-6.9094.csv,4.7385837,20.69863
stations/JAEN_SP_LAT37.7778_LON-3.8075.csv,1.6807871,20.04932


In [61]:
for (fnm in list.files("stations", full.names = T)) {
    df <- read.csv(fnm, col.names = c("date", "pr"))
    df <- df[!is.na(df$pr),]
    
    stn <- strsplit(fnm, "/|_SP")[[1]][2]
    if (as.numeric(strsplit(fnm, "LAT|_LON")[[1]][2]) > 39.5) { rnm <- "n" } else { rnm <- "s" }
    
    df$year <- as.integer(substr(df$date,1,4))
    df$month <- as.integer(substr(df$date,6,7))
    
    df <- merge(gmst, df)
    
    qq <- quantile(df[(df$year >= 1990) & (df$year <= 2020), "pr"], c(0.95, .99, .995, .99725, .9975, .999))
    
    # get abs values of extremes
    df_x <- df[df[,"pr"] >= qq["99%"],]
    
    # decluster using built-in method
    df_dc <- decluster(df[,"pr"], threshold = qq["99%"], r = 1)
    
    n_peryear <- aggregate(df[,"pr"] >= qq["99%"], by = list("year" = df$year), FUN = "sum", simplify = T)
    n_perymonth <- aggregate(df[,"pr"] >= qq["99%"], by = list(df$month, df$year), FUN = "sum", simplify = T)
    n_permonth <- aggregate(n_perymonth[,"x",drop = F], by = list("cmonth" = n_perymonth$Group.1), FUN = "mean")
    n_permonth$m_offset <- ((n_permonth$cmonth + 4) %% 12) + 1 # adjust months to cut at start of August (driest part of year)
    
    png(paste0("stn-fig/q-exceedances_",rnm,"_",stn,".png"), h = png_res, w = png_res * 3); {
        prep_window(c(1,3), oma = c(0,0,2,0))
    
        plot(df_x$year, df_x[,"pr"], main = "Magnitude of exceedances")
        lines(df_x$year, fitted(loess(pr ~ year, df_x)), col = "forestgreen", lty = "22", lwd = 3) # add a smoother through magnitude of exceedances
        lines(df_x$year, fitted(lm(pr ~ gmst, df_x)), lwd = 3, col = "darkblue")
        
        
        plot(n_peryear, main = "Number of exceedances per year", xlab = "Year")
        lines(n_peryear$year, fitted(loess(x ~ year, n_peryear)), col = "forestgreen", lty = "22", lwd = 3) # add a smoother through number of exceedances
        lines(n_peryear$year, fitted(loess(x ~ year, n_peryear)), col = "forestgreen", lty = "22", lwd = 3) # add a smoother through number of exceedances
        lines(n_peryear$year, fitted(lm(x ~ gmst, merge(gmst, n_peryear))), lwd = 3, col = "darkblue")
        
        mtext(paste0(stn, " (",rnm, ")"), outer = T, side = 3)
        
        # check that we're capturing all the exceedances
        plot(n_permonth$cmonth, n_permonth$x, main = "Mean exceedances per calendar month", xlim = c(1,12), xlab = "Calendar month", type = "o")
    }; dev.off()
}

## Quick trend fitting

In [31]:
cov_2026 <- gmst[gmst$year == 2026,,drop = F]
cov_cf <- cov_2026 - 1.3

stn_res <- data.frame(t(sapply(list.files("ts-stn", pattern = "rx1day-ondjfm", full.names = T), function(fnm) {
    stn <- strsplit(fnm, "ondjfm_|_SP")[[1]][2]
    lat <- as.numeric(strsplit(fnm, "LAT|_LON")[[1]][2])
    lon <- as.numeric(strsplit(fnm, "LON|.csv")[[1]][2])
    if (lat > 39.5) { rnm <- "n" } else { rnm <- "s" }
    df <- merge(gmst, read.csv(fnm, col.names = c("year", "pr")))
    mdl <- fit_ns("gev", "fixeddisp", df, covnm = "gmst", varnm = "pr", lower = F)
    c(stn = stn, rnm = rnm, lon = lon, lat = lat, mdl_ests(mdl, cov_f = cov_2026, cov_cf = cov_cf), nobs = nrow(df))
})))
write.csv(stn_res, "stn-res_no-bootstrap.csv")

In [17]:
stn_res[order(stn_res$lat),]

,stn,rnm,lon,lat,mu0,sigma0,alpha_gmst,shape,disp,event_magnitude,return_period,PR,dI_abs,dI_rel,aic,nobs
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
ts-stn/rx1day-ondjfm_KENITRA_PORT_LYAUTEY_SP_LAT34.3_LON-6.6.csv,KENITRA_PORT_LYAUTEY,s,-6.6,34.3,47.0929764314773,17.1923383497546,5.25171776126108,0.0397303693965906,0.365072239058203,65.8,3.4301967783491,1.64019193396355,8.88002411450774,15.6008922638547,214.613005067062,24
ts-stn/rx1day-ondjfm_TARIFA_SP_LAT36.0153_LON-5.5975.csv,TARIFA,s,-5.5975,36.0153,46.5116535741147,18.0152021952749,-8.03908272203567,-0.0134165954684124,0.387326633454738,62.4,2.96254252222044,0.598082717637423,-15.7208911553001,-20.1237990540172,284.43288570392,30
ts-stn/rx1day-ondjfm_CADIZ_SP_LAT36.5008_LON-6.2567.csv,CADIZ,s,-6.2567,36.5008,36.2386825538179,13.022181122828,-10.7169677575569,-0.0705574344638449,0.359344772081293,53.4,4.50854510883669,0.352733373011161,-25.0344630894242,-31.9176827421921,451.16916617323,51
ts-stn/rx1day-ondjfm_ROTA_SP_LAT36.6389_LON-6.3317.csv,ROTA,s,-6.3317,36.6389,42.8978491416727,13.1676758141782,-2.84834343206233,0.269939344543148,0.306954219795291,46.6,1.87421415933471,0.838450268727821,-4.20111515138581,-8.26973017987225,138.520106154875,15
ts-stn/rx1day-ondjfm_MALAGA_AEROPUERTO_SP_LAT36.6667_LON-4.4881.csv,MALAGA_AEROPUERTO,s,-4.4881,36.6667,47.1914453840205,19.1422434333426,-4.43948254102058,0.0870859167505965,0.405629521994348,73.3,4.1514154721828,0.714024386362935,-9.53549537225841,-11.5113639743523,736.850363911506,78
ts-stn/rx1day-ondjfm_JEREZ_DE_LA_FRONTERA_SP_LAT36.7506_LON-6.0556.csv,JEREZ_DE_LA_FRONTERA,s,-6.0556,36.7506,46.9255388354736,14.5478809357242,2.83288487754525,0.0476467054723274,0.310020540983679,29.1,1.03005512800026,1.02311784946842,2.1964715138154,8.16425070393023,261.75216610594,30
ts-stn/rx1day-ondjfm_GRAZALEMA_SP_LAT36.7583892_LON-5.366074.csv,GRAZALEMA,s,-5.366074,36.7583892,135.073388189809,43.1845556301978,16.1618476112845,-0.0283733969751991,0.319711796742032,102.6,1.13883765798983,1.1581416015798,14.7799456026924,16.8298069320549,219.159052797008,20
ts-stn/rx1day-ondjfm_MORON_DE_LA_FRONTERA_SP_LAT37.1581_LON-5.6156.csv,MORON_DE_LA_FRONTERA,s,-5.6156,37.1581,37.8576864860279,13.4936137040928,-3.36864620060067,0.0804211209461336,0.356429960638849,92.5,33.7997182936817,0.567547489639467,-11.3435067142278,-10.923655289728,575.685829438687,66
ts-stn/rx1day-ondjfm_GRANADA_AEROPUERTO_SP_LAT37.1894_LON-3.7892.csv,GRANADA_AEROPUERTO,s,-3.7892,37.1894,32.1860139288473,9.92007738812281,9.30539661196368,-0.0201795450809476,0.308210808895219,82.6,213.632453569336,83.2901835245527,25.8779674016155,45.6224260947103,379.673743671614,51


### And now, with NAO

In [32]:
# load covariate data
gmst = read.table("ts-obs/gmst.dat", col.names = c("year", "gmst"))
gmst$gmst <- gmst$gmst - gmst$gmst[gmst$year == 2026]

nao <- load_ts("ts-obs/med-storms_nao-djf_era5-stn.dat", col.names = c("year", "nao"))
cov_df <- merge(gmst, nao)

# define factual & counterfactual climates
cov_f <- cov_df[cov_df$year == 2026,c(-1),drop = F]
cov_cf <- rbind("pi" = cov_f - c(1.3,0),
                "naoneutral" = c(cov_f$gmst, 0),
                "pineutral" = c(-1.3, 0))


stn_res <- data.frame(t(sapply(list.files("ts-stn", pattern = "rx1day-ondjfm", full.names = T), function(fnm) {
    stn <- strsplit(fnm, "ondjfm_|_SP")[[1]][2]
    lat <- as.numeric(strsplit(fnm, "LAT|_LON")[[1]][2])
    lon <- as.numeric(strsplit(fnm, "LON|.csv")[[1]][2])
    if (lat > 39.5) { rnm <- "n" } else { rnm <- "s" }
    df <- merge(cov_df, read.csv(fnm, col.names = c("year", "pr")))
    mdl <- fit_ns("gev", "fixeddisp", df, covnm = c("nao", "gmst"), varnm = "pr", lower = F)
    c(stn = stn, rnm = rnm, lon = lon, lat = lat, mdl_ests(mdl, cov_f = cov_f, cov_cf = cov_cf), nobs = nrow(df))
})))

write.csv(stn_res, "stn-res_with-nao_no-bootstrap.csv")

# Table of stations

In [36]:
stn_res <- read.csv("stn-res_with-nao_no-bootstrap.csv")
stn_res <- stn_res[order(stn_res$rnm),]
stn_res <- stn_res[stn_res$nobs >= 20,]

In [39]:
stn_res <- stn_res[,c("rnm", "stn", "lon", "lat", "dI_rel_pi", "nobs")]

In [40]:
data.frame(stn_res$rnm, stn_res$stn, apply(round(stn_res[,c("lon", "lat")], 2), 1, paste0, collapse = ", "), round(stn_res$dI_rel, 2), stn_res$nobs)

,stn_res.rnm,stn_res.stn,apply.round.stn_res...c..lon....lat.....2...1..paste0..collapse........,round.stn_res.dI_rel..2.,stn_res.nobs
,<chr>,<chr>,<chr>,<dbl>,<int>
1,n,A_CORUNA_ALVEDRO,"-8.37, 43.31",-34.18,25
2,n,A_CORUNA,"-8.42, 43.37",18.23,38
13,n,LUGO_ROZAS,"-7.46, 43.12",36.96,26
17,n,PONTEVEDRA,"-8.62, 42.44",49.24,37
18,n,PORTO,"-8.6, 41.13",12.63,61
20,n,SANTIAGO_DE_COMPOSTELA_LABACOL,"-8.41, 42.89",43.60,33
23,n,VIGO_PEINADOR,"-8.62, 42.24",10.35,52
3,s,BADAJOZ_TALAVERA_LA_REAL,"-6.83, 38.88",-13.09,68
4,s,CADIZ,"-6.26, 36.5",-26.48,51


1              2             12             15             16 
"-8.37, 43.31" "-8.42, 43.37" "-7.46, 43.12" "-8.62, 42.44"  "-8.6, 41.13" 
            17             20              3              4              5 
"-8.41, 42.89" "-8.62, 42.24" "-6.83, 38.88"  "-6.26, 36.5" "-3.92, 38.99" 
             6              7              8              9             10 
"-4.85, 37.84" "-3.79, 37.19" "-5.37, 36.76" "-6.91, 37.28" "-3.81, 37.78" 
            11             13             14             18             19 
"-6.06, 36.75" "-4.49, 36.67" "-5.62, 37.16" "-5.88, 37.42"  "-5.6, 36.02"